# 03 Criteria Weighting

Objective criteria weights (Entropy Weight Method and CRITIC) for one of the MCDM matrices built in 02_Matrix_Preparation, and a presence-background logistic regression that checks the influence of each criterion on the location of the existing hydrogen projects. The matrix is chosen by scenario, alpha and resolution.

In [ ]:
from types import SimpleNamespace
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
from scipy.spatial import cKDTree
from scipy.special import expit
from scipy.stats import norm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from statsmodels.stats.outliers_influence import variance_inflation_factor

from gis_mcdm import (choose_run, crs_label, find_runs, load_config, normalized, read_matrix_info,
                      study_area_file, transformed)

cfg, P = load_config()  # config.toml
DATA_DIR, POLYGON_DIR, WEIGHTS_DIR, FIGURES_DIR = P.data, P.study_area, P.weights, P.figures
for folder in (WEIGHTS_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Figures are drawn at their printed size (letter page, 6.5 in text width) with 9 pt Palatino.
# For Word, insert the saved .svg (vector) or .png (600 dpi) file at 100 % so the text stays at 9 pt.
TEXT_WIDTH = 6.5
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Palatino Linotype", "Palatino", "Book Antiqua", "TeX Gyre Pagella", "DejaVu Serif"],
    "mathtext.fontset": "custom", "mathtext.rm": "serif",
    "font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.dpi": 200,  # sharper preview in the notebook
    "savefig.dpi": 600, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
})


def save_figure(fig, name):
    for ext in ("png", "svg"):
        fig.savefig(FIGURES_DIR / f"{name}.{ext}")

In [ ]:
# Matrix to analyse (press Enter to keep the default shown in brackets: the run of [run] in config.toml)
run = choose_run(cfg, find_runs(cfg, P))
MATRIX_PATH, SCENARIO, ALPHA_TAG, RES_TAG, RUN_TAG = run.matrix, run.scenario, run.alpha, run.res, run.tag

WEIGHTS_PATH = WEIGHTS_DIR / f"Criteria_Weights_{RUN_TAG}.csv"
ANCHOR_PATH = POLYGON_DIR / f"anchor_grid_{RES_TAG}.tif"  # grid of the matrix, used to place the projects
print(f"\nRun: {SCENARIO}, alpha {ALPHA_TAG}, {RES_TAG} ({MATRIX_PATH.name})")

## 1. Criteria weights

In [ ]:
raw_df = pd.read_csv(MATRIX_PATH)
CRIT = {c["column"]: c for c in read_matrix_info(MATRIX_PATH)}  # criteria of the matrix, written by 02
crit_cols = list(CRIT)
LABELS = {col: c["label"] for col, c in CRIT.items()}

# 0-1 scale: transform (e.g. log), costs inverted, benefits min-max; scaled criteria unchanged
norm_matrix = np.column_stack([normalized(raw_df[col], CRIT[col]) for col in crit_cols])

In [ ]:
def ewm_weights(matrix, eps=1e-9):
    """Entropy Weight Method (eps avoids log(0) on min-max scaled zeros); one column at a time to save memory."""
    entropy = np.empty(matrix.shape[1])
    for j in range(matrix.shape[1]):
        p = matrix[:, j] + eps
        p /= p.sum()
        entropy[j] = -np.sum(p * np.log(p)) / np.log(len(matrix))
    divergence = 1.0 - entropy
    return divergence / divergence.sum()


def critic_weights(matrix):
    """CRITIC: standard deviation x sum of (1 - correlation) with the other criteria."""
    info = matrix.std(axis=0, ddof=1) * (1.0 - np.corrcoef(matrix, rowvar=False)).sum(axis=1)
    return info / info.sum()


weights_df = pd.DataFrame({"Criterion": crit_cols, "EWM_Weight": ewm_weights(norm_matrix),
                           "CRITIC_Weight": critic_weights(norm_matrix)})
weights_df.to_csv(WEIGHTS_PATH, index=False)
del norm_matrix

display(weights_df.sort_values("EWM_Weight", ascending=False).style.format({"EWM_Weight": "{:.4f}", "CRITIC_Weight": "{:.4f}"}))

In [ ]:
plot_df = (weights_df.assign(Label=weights_df["Criterion"].map(LABELS))
           .sort_values("EWM_Weight").set_index("Label")[["EWM_Weight", "CRITIC_Weight"]])

fig, ax = plt.subplots(figsize=(TEXT_WIDTH, 2.0), layout="constrained")
plot_df.plot.barh(ax=ax, color=["#1F8621", "#0E5BA8"], width=0.8, edgecolor="black", linewidth=0.2)
ax.set_xlabel("Weight")
ax.set_ylabel("")
ax.grid(axis="x", linestyle=":", linewidth=0.5, alpha=0.7)
ax.set_axisbelow(True)
ax.legend(["EWM", "CRITIC"], loc="lower right", frameon=False)
sns.despine(ax=ax)

save_figure(fig, f"Criteria_Weights_Comparison_{RUN_TAG}")
plt.show()

## 2. Logistic regression: influence of the criteria on existing projects

Skipped if `[validation] projects` in config.toml is empty or missing, or if too few projects lie in the candidate area.

In [ ]:
RANDOM_SEED = 42
THINNING_KM = 10          # minimum distance between presences
BACKGROUND_RATIO = 10     # background sites per presence
BACKGROUND_BUFFER_KM = 5  # minimum distance from a background site to any project
N_BOOTSTRAP = 200         # bootstrap replicates
CV_BLOCK_KM = 250         # side of the spatial cross-validation blocks
CV_FOLDS = 5              # spatial cross-validation folds

rng = np.random.default_rng(RANDOM_SEED)

# The regression needs the existing projects ([validation] projects in config.toml)
RUN_LR = P.projects is not None and P.projects.exists()
if not RUN_LR:
    print("No projects layer ([validation] projects in config.toml): logistic regression skipped.")

# z-score parameters from ALL candidate cells, on the same scale as the MCDM aggregation (transform, e.g. log)
z_mean, z_std = {}, {}
for col in crit_cols:
    values = transformed(raw_df[col], CRIT[col])
    z_mean[col], z_std[col] = values.mean(), values.std(ddof=1)


def design_matrix(idx):
    """Intercept + standardized criteria for the given matrix rows."""
    rows = raw_df.loc[idx]
    X = pd.DataFrame({col: (transformed(rows[col], CRIT[col]) - z_mean[col]) / z_std[col] for col in crit_cols})
    X.insert(0, "Intercept", 1.0)
    return X


def fit_logit(X, y, max_iter=100, tol=1e-8):
    """Firth's penalized maximum-likelihood logistic regression (Firth 1993; Heinze & Schemper 2002).
    The Jeffreys-prior penalty 0.5 log|I(b)| keeps the estimates finite and reduces small-sample bias when the
    few presences are (quasi-)separated from the background, where ordinary maximum likelihood diverges.
    Returns None only if the information matrix is singular."""
    A, yv = np.asarray(X, dtype=float), np.asarray(y, dtype=float)

    def penalized(beta):
        eta = A @ beta
        p = expit(eta)
        info = A.T @ (A * (p * (1 - p))[:, None])
        loglik = -np.sum(yv * np.logaddexp(0, -eta) + (1 - yv) * np.logaddexp(0, eta))
        return loglik + 0.5 * np.linalg.slogdet(info)[1], loglik, p, info

    try:
        beta = np.zeros(A.shape[1])
        pl, loglik, p, info = penalized(beta)
        for _ in range(max_iter):
            cov = np.linalg.inv(info)
            h = p * (1 - p) * np.einsum("ij,jk,ik->i", A, cov, A)    # leverages of the weighted hat matrix
            step = cov @ (A.T @ (yv - p + h * (0.5 - p)))            # Firth-modified score
            for _ in range(30):                                      # step halving: penalized likelihood must not drop
                candidate = penalized(beta + step)
                if candidate[0] >= pl - 1e-12:
                    break
                step /= 2
            beta = beta + step
            pl, loglik, p, info = candidate
            if np.abs(step).max() < tol:
                break
        se = np.sqrt(np.diag(np.linalg.inv(info)))
    except np.linalg.LinAlgError:
        return None

    p0 = yv.mean()
    loglik_null = len(yv) * (p0 * np.log(p0) + (1 - p0) * np.log(1 - p0))
    return SimpleNamespace(
        params=pd.Series(beta, index=X.columns),
        bse=pd.Series(se, index=X.columns),
        pvalues=pd.Series(2 * norm.sf(np.abs(beta / se)), index=X.columns),  # Wald test
        prsquared=1 - loglik / loglik_null,                                  # McFadden R²
        predict=lambda X_new: expit(np.asarray(X_new, dtype=float) @ beta),
    )

In [ ]:
# 1-2. Presences: project cells inside the candidate area, one per cell, thinned to THINNING_KM
if RUN_LR:
    projects = gpd.read_file(P.projects)
    with rasterio.open(ANCHOR_PATH) as src:
        grid_crs, grid_transform, (grid_h, grid_w) = src.crs, src.transform, src.shape
    projects = projects.to_crs(grid_crs)
    xy = np.column_stack([projects.geometry.x, projects.geometry.y])

    # Matrix row of the cell containing each project (matrix rows are sorted by cell, so a binary search suffices)
    rows, cols = map(np.asarray, rasterio.transform.rowcol(grid_transform, xy[:, 0], xy[:, 1]))
    cell_keys = raw_df["Row_Idx"].to_numpy(np.int64) * grid_w + raw_df["Col_Idx"].to_numpy(np.int64)
    project_keys = rows.astype(np.int64) * grid_w + cols
    pos = np.searchsorted(cell_keys, project_keys).clip(0, len(cell_keys) - 1)
    inside = (rows >= 0) & (rows < grid_h) & (cols >= 0) & (cols < grid_w) & (cell_keys[pos] == project_keys)
    projects["Matrix_Idx"] = np.where(inside, pos, -1)
    projects["Status"] = np.select([~inside, projects.duplicated("Matrix_Idx").to_numpy()],
                                   ["Outside candidate area", "Same cell as another project"], "Presence")

    # Spatial thinning: in a random order, keep a presence only if it is >= THINNING_KM from those already kept
    candidates = projects.index[projects["Status"] == "Presence"]
    kept = []
    for i in rng.permutation(candidates):
        if all(np.hypot(*(xy[i] - xy[j])) >= THINNING_KM * 1000 for j in kept):
            kept.append(i)
    projects.loc[candidates.difference(kept), "Status"] = f"Removed by thinning (< {THINNING_KM} km)"
    presence_idx = projects.loc[projects["Status"] == "Presence", "Matrix_Idx"].to_numpy()

    if len(presence_idx) < 2 * CV_FOLDS:
        print(f"Only {len(presence_idx)} presences after thinning (at least {2 * CV_FOLDS} needed): "
              "logistic regression skipped.")
        RUN_LR = False

In [ ]:
# 3. Background sites: simple random sample of candidate cells >= BACKGROUND_BUFFER_KM from any project
if RUN_LR:
    dist_to_project, _ = cKDTree(xy).query(raw_df[["X_Coord", "Y_Coord"]].to_numpy())
    background_pool = raw_df.index.to_numpy()[dist_to_project >= BACKGROUND_BUFFER_KM * 1000]
    background_idx = rng.choice(background_pool, size=BACKGROUND_RATIO * len(presence_idx), replace=False)

    sample_idx = np.concatenate([presence_idx, background_idx])
    y = np.concatenate([np.ones(len(presence_idx)), np.zeros(len(background_idx))])

In [ ]:
# 4-5. Predictor scaling and coefficient estimation
if RUN_LR:
    X = design_matrix(sample_idx)
    model = fit_logit(X, y)
    if model is None:
        raise RuntimeError("Singular information matrix: check the predictors for constant or duplicated columns.")

    coef_df = pd.DataFrame({
        "Term": X.columns,
        "Beta_Std": model.params.to_numpy(),
        "SE": model.bse.to_numpy(),
        "p_value": model.pvalues.to_numpy(),
        "Odds_Ratio": np.exp(model.params.to_numpy()),
        "VIF": [np.nan] + [variance_inflation_factor(X.to_numpy(), i) for i in range(1, X.shape[1])],
    })

In [ ]:
# 6. Uncertainty (bootstrap) and 8. contribution of each criterion
def contribution(beta):
    """C_j = |beta_j| / sum_k |beta_k| x 100 (criteria only, without the intercept)."""
    return 100 * beta[crit_cols].abs() / beta[crit_cols].abs().sum()


if RUN_LR:
    boot_beta = []
    for _ in range(N_BOOTSTRAP):
        idx_b = np.concatenate([rng.choice(presence_idx, size=len(presence_idx), replace=True),
                                rng.choice(background_pool, size=len(background_idx), replace=False)])
        if (model_b := fit_logit(design_matrix(idx_b), y)) is not None:
            boot_beta.append(model_b.params)
    boot_beta = pd.DataFrame(boot_beta)

    boot_contrib = boot_beta.apply(contribution, axis=1)
    coef_df["Beta_CI_Low"] = boot_beta.quantile(0.025).to_numpy()
    coef_df["Beta_CI_High"] = boot_beta.quantile(0.975).to_numpy()
    coef_df["Contribution_pct"] = [np.nan] + list(contribution(model.params))
    coef_df["Contribution_CI_Low"] = [np.nan] + list(boot_contrib.quantile(0.025))
    coef_df["Contribution_CI_High"] = [np.nan] + list(boot_contrib.quantile(0.975))

    display(coef_df[coef_df["Term"] != "Intercept"].set_index("Term").sort_values("Contribution_pct", ascending=False).round(3))

In [ ]:
# 7. Model validation: ROC-AUC on the full sample and with spatial block cross-validation
if RUN_LR:
    sample_xy = raw_df.loc[sample_idx, ["X_Coord", "Y_Coord"]].to_numpy()
    # Blocks of CV_BLOCK_KM; halved until there are at least two blocks per fold (small study areas)
    block_km = CV_BLOCK_KM
    while True:
        blocks = pd.Series(list(map(tuple, np.floor(sample_xy / (block_km * 1000)).astype(int)))).factorize()[0]
        if blocks.max() + 1 >= 2 * CV_FOLDS or block_km < 1:
            break
        block_km /= 2
    if block_km != CV_BLOCK_KM:
        print(f"Spatial CV blocks reduced to {block_km:g} km to have enough blocks for {CV_FOLDS} folds")
    cv = StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

    cv_pred = np.full(len(y), np.nan)
    fold_auc = []
    for train, test in cv.split(X, y, groups=blocks):
        cv_pred[test] = fit_logit(X.iloc[train], y[train]).predict(X.iloc[test])
        fold_auc.append(roc_auc_score(y[test], cv_pred[test]))

    status = projects["Status"].value_counts()
    print(f"{len(projects)} projects: {status.get('Outside candidate area', 0)} outside the candidate area, "
          f"{status.get('Same cell as another project', 0)} in a shared cell, "
          f"{status.filter(like='thinning').sum()} removed by thinning -> {len(presence_idx)} presences + "
          f"{len(background_idx)} background sites (pool {len(background_pool):,})")
    print(f"McFadden R² {model.prsquared:.3f} | max VIF {coef_df['VIF'].max():.2f} | bootstrap {len(boot_beta)}/{N_BOOTSTRAP} | "
          f"AUC full sample {roc_auc_score(y, model.predict(X)):.3f}, spatial CV {roc_auc_score(y, cv_pred):.3f} "
          f"(folds {np.mean(fold_auc):.3f} ± {np.std(fold_auc):.3f})")

In [ ]:
if RUN_LR:
    plot_df = coef_df[coef_df["Term"] != "Intercept"].sort_values("Contribution_pct")
    ypos = np.arange(len(plot_df))

    fig, ax = plt.subplots(figsize=(TEXT_WIDTH, 2.0), layout="constrained")
    ax.barh(ypos, plot_df["Contribution_pct"], height=0.65, color="#1F8621", edgecolor="black", linewidth=0.4)
    ax.set_yticks(ypos, [LABELS[t] for t in plot_df["Term"]])
    ax.set_xlabel("Contribution (%)")
    ax.grid(axis="x", linestyle=":", linewidth=0.5, alpha=0.7)
    ax.set_axisbelow(True)
    sns.despine(ax=ax)

    save_figure(fig, f"LR_Contribution_{RUN_TAG}")
    plt.show()

In [ ]:
if RUN_LR:
    # Outline simplified to 1 km (below one printed pixel) to keep the SVG light
    study_area = gpd.read_file(study_area_file(P)).to_crs(grid_crs).simplify(1000)
    bg_xy = raw_df.loc[background_idx, ["X_Coord", "Y_Coord"]].to_numpy()
    used = (projects["Status"] == "Presence").to_numpy()

    # Sample exported for QGIS: GeoTIFF on the analysis grid (0 = NoData, 1 = background, 2 = presence)
    # and the same sites as points (GeoPackage), easier to style as symbols
    LR_DIR = P.lr
    LR_DIR.mkdir(exist_ok=True)
    sample_arr = np.zeros((grid_h, grid_w), dtype=np.uint8)
    sample_cells = raw_df.loc[sample_idx, ["Row_Idx", "Col_Idx"]].to_numpy(int)
    sample_arr[sample_cells[:, 0], sample_cells[:, 1]] = np.where(y == 1, 2, 1)
    with rasterio.open(LR_DIR / f"LR_Sample_{RUN_TAG}.tif", "w", driver="GTiff", height=grid_h, width=grid_w, count=1,
                       dtype="uint8", nodata=0, crs=grid_crs, transform=grid_transform, compress="deflate") as dst:
        dst.write(sample_arr, 1)
    gpd.GeoDataFrame({"Type": ["Background"] * len(bg_xy) + ["Presence"] * int(used.sum())},
                     geometry=gpd.points_from_xy(np.r_[bg_xy[:, 0], xy[used, 0]], np.r_[bg_xy[:, 1], xy[used, 1]]),
                     crs=grid_crs).to_file(LR_DIR / f"LR_Sample_{RUN_TAG}.gpkg")

    fig, ax = plt.subplots(figsize=(TEXT_WIDTH, 3), layout="constrained")
    study_area.plot(ax=ax, facecolor="#f0f0f0", edgecolor="black", linewidth=0.4)
    ax.scatter(bg_xy[:, 0], bg_xy[:, 1], s=3, color="#0E5BA8", alpha=0.6, linewidths=0, label="Background")
    ax.scatter(xy[used, 0], xy[used, 1], s=18, marker="^", color="#d7191c", edgecolor="black", linewidths=0.4,
               label="Presences")
    ax.legend(loc="best", frameon=False, markerscale=1.5)
    km = FuncFormatter(lambda v, _: f"{v / 1000:,.0f}")
    ax.xaxis.set_major_formatter(km)
    ax.yaxis.set_major_formatter(km)
    ax.set_xlabel(f"Easting (km), {crs_label(grid_crs)}")
    ax.set_ylabel(f"Northing (km), {crs_label(grid_crs)}")

    save_figure(fig, f"LR_Sample_Map_{RUN_TAG}")
    plt.show()